# Task 4 Optuna screening
Attach FS2K as a private dataset. Run in order. Eight five-epoch trials are provisional screening, not final training. Download the complete backup before ending a session. No Google Drive saving is performed.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
candidates = [p.parent for p in Path('/kaggle/input').rglob('anno_train.json') if (p.parent/'photo').is_dir() and (p.parent/'sketch').is_dir()]
print(candidates)
assert len(candidates)==1, 'Attach one FS2K dataset or set raw to its exact root.'
raw = candidates[0]
subprocess.run([sys.executable,'-m','scripts.prepare_fs2k','--raw-dir',str(raw)],cwd=repo,check=True)


In [ ]:
import shutil
search = Path('/kaggle/working/task4-optuna')
resume_search_folder = None  # Set attached folder containing study.json after a reset.
if resume_search_folder is not None and not search.exists():
    source = Path(resume_search_folder)
    assert (source/'study.json').is_file(), 'Use complete search backup, not baseline folder'
    shutil.copytree(source, search)
subprocess.run([sys.executable, '-u', '-m', 'optimization.optimize_task4',
    '--raw-dir', str(raw), '--output-dir', str(search), '--device', 'cuda',
    '--trials', '8', '--screen-epochs', '5', '--max-hours', '2'], cwd=repo, check=True)


In [ ]:
import json
from IPython.display import display, FileLink
if (search/'leaderboard.json').exists():
    board=json.loads((search/'leaderboard.json').read_text())
    for row in board['ranked_trials']:
        print(row['number'], row['score'], row['params'])
else:
    print('No completed trial yet; save backup and rerun search to resume.')
backup=shutil.make_archive('/kaggle/working/task4-optuna_backup','zip',root_dir=search.parent,base_dir=search.name)
display(FileLink(backup))


Share the search backup before selecting the final run. Rerunning the search cell continues pending trials; --trials is the total target. After reviewing metrics and previews, retrain selected_config.yaml from scratch in a NEW folder for its full 30 epochs, without --resume or --stop-after-epoch. Do not treat screening metrics as final evidence.